**Business Problem :** Predict Sales Price for each house from the Ames Housing datset (expanded version of the often cited Boston Housing dataset)

**Statistical Problem:** Which features impact to the price of a house the most and drive buying decisions. Need to analyze numeric features like area, distance from road along with categorical features of a housing project like neighbourhood etc to determine the right set of creative features determining the sales price of a house.

**What are we Predicting**: Sales Price of each house.

**Is this Regression or Classification**: Its a regression problem since the target variables is a numeric feature.

**Metric Used by Kaggle:** Submissions are evaluated on Root-Mean-Squared-Error (RMSE) between the logarithm of the predicted value and the logarithm of the observed sales price. (Taking logs means that errors in predicting expensive houses and cheap houses will affect the result equally.)

In [1]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)
print(f"Working From: {project_root}")

Working From: c:\Users\ChandrG\ML-Project-House-Prices


In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from src import config

print("Config Values:")
for k, v in vars(config).items():
    if not k.startswith("__"):
        print(f"{k}: {v}")

Config Values:
os: <module 'os' (frozen)>
PROJECT_ROOT: c:\Users\ChandrG\ML-Project-House-Prices
DATA_DIR: c:\Users\ChandrG\ML-Project-House-Prices\data
RAW_DATA_DIR: c:\Users\ChandrG\ML-Project-House-Prices\data\raw
PROCESSED_DATA_DIR: c:\Users\ChandrG\ML-Project-House-Prices\data\processed
MODELS_DIR: c:\Users\ChandrG\ML-Project-House-Prices\models
RESULTS_DIR: c:\Users\ChandrG\ML-Project-House-Prices\results
NOTEBOOKS_DIR: c:\Users\ChandrG\ML-Project-House-Prices\notebooks
TRAIN_DATA_PATH: c:\Users\ChandrG\ML-Project-House-Prices\data\raw\train.csv
TEST_DATA_PATH: c:\Users\ChandrG\ML-Project-House-Prices\data\raw\test.csv
DATA_DESCRIPTION_PATH: c:\Users\ChandrG\ML-Project-House-Prices\data\raw\data_description.txt
SAMPLE_SUBMISSION_PATH: c:\Users\ChandrG\ML-Project-House-Prices\data\raw\sample_submission.csv


In [3]:
df_train = pd.read_csv(config.TRAIN_DATA_PATH)
print(f"Shape of df_train: {df_train.shape}")
print("\nInfo of df_train:")
df_train.info()
print("\nDescription of df_train:")
df_train.describe().T

Shape of df_train: (1460, 81)

Info of df_train:
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual   

,count,mean,std,min,25%,50%,75%,max
Id,1460.0,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,1460.0,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,1201.0,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,1460.0,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1460.0,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,1460.0,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1460.0,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1460.0,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,1452.0,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,1460.0,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


In [ ]:
# Check for Categorical Columns
cat_cols = df_train.select_dtypes(include='str').columns
print(f"\nCategorical Columns: {cat_cols}")

Index(['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
       'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
       'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='str')

In [4]:
# Check for missing values
print("\nMissing values in df_train:")
df_train.filter(items=[col for col in df_train.columns if df_train[col].isnull().any()]).isnull().sum()


Missing values in df_train:


LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

**Startegy for missing feature's statistical imputation**
1. LotFrontage : Median LotFrontage value based on 'LotArea Bin', a created deterministic feature.
2. Alley : Fill missing values as 'NA' (No Alley Access).
3. MasVnrType : Filll missing values based on non-missing grouped results of Exterior1st aggregated by mode of MasVnrType.
4. BsmtQual : Fill missing values by "NA" (No Basement) value.
5. BsmtCond : Fill missing values by "NA" (No Basement) value.
6. BsmtExposure : Create a conditional value if BsmtQual & BsmtCond == "NA"  then 'NA' (No Basement) else 'No' (No Exposure).
7. BsmtFinType1 : Fill missing values by 'NA' (No Basement) value.
8. BsmtFinType2 : Fill missing values by condition if BsmtFinType1 == "NA" then then 'NA' (No Basement) else group by associated value from BsmtFinType1 column and take mode of the BsmtFinType2 values.
9. Electrical : Fill missing values by the mode.
10. FireplaceQu : Fill missing values by NA	(No Fireplace).
11. GarageType : Fill missing values by NA	(No Garage).
12. GarageYrBlt : Need to verify if we can have a placeholder value for these.
13. GarageFinish :  Fill missing values by NA	(No Garage).
14. GarageQual : Fill missing values by NA	(No Garage).
15. GarageCond : Fill missing values by NA	(No Garage).
16. PoolQC : Fill missing values by NA	(No Pool).
17. Fence : Fill missing values by NA	(No Fence).
18. MiscFeature : Fill missing values by NA	(None).